# LLM Playground on Isambard

Load **NVIDIA Nemotron 3 Super (120B-A12B)** across the four GH200 GPUs of a single
Isambard node and generate text with the HuggingFace `transformers` pipeline.

**Before you run this**, set the environment up — from a **login node first**
(that step installs the VS Code CLI that `tunnel.sbatch` needs), then again
inside the tunnel to verify the GPUs:

```bash
export PROJECT_STORAGE=/projects/<your-project>/<your-space>
bash setup_environment.sh     # login node
sbatch tunnel.sbatch          # then connect, and re-run setup on the node
```

Then pick this repo's `.venv` as the notebook kernel:
*Ctrl/Cmd+Shift+P -> "Notebook: Select Kernel" -> `.venv/bin/python`*.

---


## 1. Environment check

Run this first. It takes two seconds and tells you immediately whether the two
classic Isambard traps have bitten you.


In [ ]:
import os

# --- Trap 1: an inherited LD_PRELOAD ---------------------------------------
# Isambard job scripts routinely export
#     LD_PRELOAD=/tools/brics/.../libnccl.so
# as part of a copy-pasted Slingshot block. That force-loads the SYSTEM NCCL
# ahead of the copy bundled inside the torch wheel; the system copy is older
# and lacks symbols torch needs, so `import torch` dies with
# "undefined symbol: nccl...".
#
# IMPORTANT: this cannot be fixed from inside a running process. The dynamic
# loader applies LD_PRELOAD at exec time, so by the time Python is running the
# old library is already mapped into it. Deleting the variable from os.environ
# affects only CHILD processes -- not this one. (Try it: the library stays in
# /proc/self/maps.) It has to be unset in the shell BEFORE the kernel starts,
# so we detect it and stop rather than pretend to fix it.
if os.environ.get("LD_PRELOAD"):
    raise SystemExit(
        f"LD_PRELOAD is set:\n    {os.environ['LD_PRELOAD']}\n\n"
        "This breaks `import torch`, and it cannot be undone from inside a\n"
        "running kernel. Fix it in the shell, then restart the kernel:\n\n"
        "    unset LD_PRELOAD\n\n"
        "tunnel.sbatch already does this, so a kernel started inside the\n"
        "tunnel is clean. You will only hit this if the kernel was launched\n"
        "from a shell that sourced a Slingshot/NCCL block."
    )

# --- Where the model weights are cached ------------------------------------
# Point PROJECT_STORAGE at YOUR project's storage. Caching a 231 GB model in
# $HOME will blow the (small) home quota, and another project's storage is not
# readable by you. PROJECTDIR is set for you by the BriCS default modules.
PROJECT_STORAGE = (
    os.environ.get("PROJECT_STORAGE")
    or os.environ.get("PROJECTDIR")
    or os.path.expanduser("~")
)
os.environ.setdefault("HF_HOME", os.path.join(PROJECT_STORAGE, "hf"))
print(f"HF_HOME = {os.environ['HF_HOME']}")

import torch, transformers

print(f"torch          {torch.__version__}")
print(f"transformers   {transformers.__version__}")
print(f"CUDA available {torch.cuda.is_available()}")
print(f"GPUs visible   {torch.cuda.device_count()}")

# --- Trap 2: a CPU-only or CUDA-13 wheel -----------------------------------
# Isambard is ARM (aarch64). Plain `pip install torch` gives you either a
# CPU-ONLY aarch64 build (silently no GPU, and no error at all) or a CUDA-13
# build needing a newer driver than this cluster has. Both look fine until
# they aren't. pyproject.toml pins the cu126 index to avoid both.
assert "+cu" in torch.__version__, (
    f"torch {torch.__version__} has no +cuXXX suffix -- this is the CPU-only "
    "aarch64 wheel. See [tool.uv.sources] in pyproject.toml."
)
assert torch.cuda.is_available(), (
    "No GPU visible. On a login node? Login nodes have no GPUs. On a compute "
    "node? Check the job actually requested --gpus-per-node=4."
)

total = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count()))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU[{i}] {p.name}  sm_{p.major}{p.minor}  {p.total_memory/2**30:.1f} GiB")
print(f"\naggregate GPU memory: {total/2**30:.1f} GiB")


## 2. Load the model

`nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` is a **hybrid** model: 88 layers
mixing Mamba2 state-space layers, a few attention layers, and a 512-expert MoE.
In BF16 the weights are about **225 GiB**, so:

| GPUs | Aggregate memory | Fits? |
|------|-----------------|-------|
| 1    | 95 GiB          | no    |
| 2    | 190 GiB         | no    |
| 4    | 380 GiB         | yes — 249 GiB used, ~131 GiB free |

`device_map="auto"` is what makes this work. It is **not** distributed training:
it is a single process, and `accelerate` simply places different layers on
different GPUs, passing activations between them with ordinary tensor copies.
There is no process group and no NCCL collective — which is exactly why this
notebook needs no NCCL module and no Slingshot configuration.

Expect the load to take **~110 seconds** (reading ~231 GB from the shared Lustre
filesystem). You will see a warning that the Mamba2 "fast path is not
available" — that is expected and fine; see the note after the next cell.


In [ ]:
import time
from transformers import pipeline

MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

t0 = time.time()
pipe = pipeline(
    "text-generation",
    model=MODEL,
    dtype=torch.bfloat16,   # note: `dtype`, not the older `torch_dtype`
    device_map="auto",      # shard the model across all visible GPUs
)
print(f"loaded in {time.time()-t0:.1f}s")

In [ ]:
from collections import Counter

# Which GPU did each layer land on?
spread = Counter(pipe.model.hf_device_map.values())
print("layers per GPU:", dict(sorted(spread.items())))

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB used / {total/2**30:.1f} GiB")

> **About that "fast path is not available" warning.**
> The Mamba2 layers have optional hand-written CUDA kernels (`mamba-ssm`,
> `causal-conv1d`). Neither ships an ARM wheel, so installing them means
> compiling CUDA from source — slow and fragile, and deliberately out of scope
> here. `transformers` falls back to a pure-PyTorch implementation that is
> numerically correct, just slower. That is the right trade for a playground.
>
> If you later need the speed, that is when you reach for `module load cuda/12.6`
> and `export CC=/usr/bin/gcc-12` (the system `gcc` is 7.5 and too old).


## 3. Generating text

Two things about this model specifically.

**It is a reasoning model, and reasoning is ON by default.** Its chat template
opens a `<think>` block unless you say otherwise, so a short request spends its
whole token budget thinking out loud and you never see an answer. Passing
`enable_thinking=False` to `apply_chat_template` closes the block immediately.

**`enable_thinking` belongs to the chat template, not to `generate`.** Passing it
to the pipeline raises `ValueError: The following model_kwargs are not used by
the model`. So we render the prompt first, then generate from the string.


In [ ]:
def chat(user_message, max_new_tokens=128, think=False, **kw):
    """Render the chat template, then generate. Returns the assistant's reply."""
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=think,      # a CHAT TEMPLATE kwarg, not a generate kwarg
    )
    out = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,             # reply only, not the prompt back
        clean_up_tokenization_spaces=False, # destructive for BPE tokenizers
        **kw,
    )
    return out[0]["generated_text"].strip()

> **Two warnings you will see on every call. Both are benign.**
>
> * `Both max_new_tokens (=128) and max_length (=20) seem to have been set.`
>   The model ships a `generation_config` carrying the library-default
>   `max_length=20`. Our `max_new_tokens` takes precedence, exactly as the
>   message says. Setting `generation_config.max_length = None` does *not*
>   silence it -- the config object coerces the value back to its default.
> * `Passing generation_config together with generation-related arguments is
>   deprecated.` A `transformers` housekeeping notice about the pipeline API,
>   not about anything you did.
>
> Learning to read past benign warnings -- and to recognise the ones that are
> *not* benign, like the `undefined symbol` failure in section 1 -- is most of
> what makes an HPC software stack tractable.


### Example 1 — a short factual explanation

In [ ]:
t0 = time.time()
reply = chat("Explain what a supercomputer interconnect does, in two sentences.")
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

### Example 2 — something creative

In [ ]:
t0 = time.time()
reply = chat("Write a haiku about GPUs waiting in a job queue.", max_new_tokens=64)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

### Example 3 — the same prompt, with reasoning switched on

This is the interesting one. Compare the output to what you would get with
`think=False`: the model narrates its own approach inside a `<think>` block
before answering. Useful when you want to see the reasoning; wasteful when you
just want the answer.


In [ ]:
t0 = time.time()
reply = chat(
    "A job needs 12 GPUs. Each node has 4. How many nodes, and what if one node fails?",
    max_new_tokens=200,
    think=True,
)
print(reply)
print(f"\n[{time.time()-t0:.1f}s]")

## 4. Freeing the GPUs

The model holds ~225 GiB. Release it before loading anything else — or just
restart the kernel, which is more reliable.


In [ ]:
import gc

# Dropping the reference is necessary but often not sufficient: Jupyter keeps
# its own references to cell outputs, so memory may not fall to zero here.
# Restarting the kernel is the reliable way to fully release the GPUs.
del pipe
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU[{i}] {(total-free)/2**30:5.1f} GiB still resident")

print("\nIf that is still large, restart the kernel to fully free the GPUs.")

---

## Where to go next

- **Your job has a 24-hour walltime cap.** It is hard: the job dies at
  `24:00:00` and takes your tunnel with it. Save work to `/projects`, never to
  the node.
- **This notebook is single-node.** Models too large for one node (the 550B
  Nemotron Ultra, ~1.1 TB) need a multi-node serving engine such as vLLM with
  pipeline parallelism — a different exercise, and one where the Slingshot and
  NCCL configuration we skipped here suddenly matters a great deal.
- **Check for a shared model cache before downloading.** If your site
  keeps one, the model you want may already be there — and the project
  storage quota is shared across everyone using it.
